# Creating Lightcones

Rather than modelling galaxy populations at isolated epochs, synthpop can generate light-cone realizations that include galaxies distributed over a range of redshifts.

Like a `GalaxyPopulation` a `Lightcone` contains a list of `synthesizer` `Galaxy` objects. 



In [ ]:
from astropy.cosmology import Planck18 as cosmo
import numpy as np
from synthesizer.filters import FilterCollection
from synthesizer.grid import Grid
from synthpop.models import Constant, Default
from synthpop import Lightcone
from unyt import Msun, arcminute, Myr


In [ ]:
# Load grid
grid = Grid("test_grid")

# Define lightcone solid angle
solid_angle = 10 * arcminute**2

# Define model including the galaxy stellar mass function, star formation history, and metallicity history
model = Default()
model = Constant()

In [ ]:
# Create the lightcone
lc = Lightcone(
    model=model,
    minimum_stellar_mass=1E9*Msun, 
    maximum_stellar_mass=1E11*Msun,
    cosmology=cosmo,
    grid=grid,
    solid_angle=solid_angle,
    random_seed=42)

print(lc)


### Useful methods and plots

In [ ]:
# Calculate the stellar mass density
print(f'Stellar mass density = {lc.calculate_total_stellar_mass_density(redshift_range=[0,0.5]).to("Msun/Mpc**3"):<10.2e}')

In [ ]:
redshift_bins = np.arange(0, 10, 0.5)

# Calculate and plot the stellar mass density history
lc.plot_cosmic_stellar_mass_density(redshift_bins=redshift_bins)

In [ ]:
# Calculate star formation rates for all galaxies in the lightcone avearged over the last 10 Myr
lc.calculate_star_formation_rates(age=10*Myr)

# Plot the cosmic star formation rate history
lc.plot_cosmic_sfrd(redshift_bins=redshift_bins)

## Observed-frame photometry

In [ ]:

# Get the filter collection
filter_codes = [
    f"JWST/NIRCam.{f}"
    for f in ["F090W", "F150W", "F200W", "F277W", "F356W", "F444W"]
]

filters = FilterCollection(
    filter_codes=filter_codes,
    new_lam=grid.lam,
)

from synthesizer.emission_models import IncidentEmission

incident = IncidentEmission(grid=grid)

lc.generate_spectra(incident)

lc.generate_photometry("incident", filters)

In [ ]:

lc.plot_number_counts("JWST/NIRCam.F200W")

In [ ]:
lc.plot_number_counts("JWST/NIRCam.F150W", magnitude=True)

In [ ]:
import h5py
import matplotlib.pyplot as plt

band = "JWST/NIRCam.F444W"
with h5py.File("/Users/jt458/flags_counts/flags_galaxy_counts_v3105.hdf5", "r") as f:
    f.visit(print)

    bin_edges = np.arange(start=9.75, stop=32.25, step=0.5)
    centres = (bin_edges[1:] + bin_edges[:-1]) / 2
    widths = np.diff(bin_edges)

    n = (10**f[band]["fiducial"]["N"][:]) / widths


obs = {"x": centres, "phi": n}
lc.plot_number_counts(band, magnitude=True, observations=obs, bin_edges=bin_edges)
